# Demonstration of Luma Module 
This notebook contain the implementation of the source code for each module in the EPISTEM land cover mapping framework

## Library import and earth engine initialization
If you have earth engine account you could used that to authenticate and initialize the earth engine. However, if you did not have the account, service account initialization is avaliable

In [1]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ee 
import luma_ge

#Option 1: Manual authenticate using personal account
#Instructions for manual authentication
luma_ge.print_auth_instructions()
#uncomment the below line and follow earth engine authentication process
luma_ge.authenticate_manually()

#Option 2: Autheticate using service account (json file)
#service_account_path = '../auth/earth-engine-451407-520c1ef64879.json' #(if failed use this)
service_account_path = '../auth/ee-epstm2024.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")


    EARTH ENGINE AUTHENTICATION NOTES:
    
    1. Make sure you already have a google cloud project that has enable the Earth Engine API and registered to 
       commercial or non-commercial use. For more information visit: https://developers.google.com/earth-engine/guides/access 
    
    2. you can authenticate programmatically by calling: from luma_ge.ee_config import authenticate_manually
       authenticate_manually()
    
    3. This will open a web browser. Sign in with your Google account that has Earth Engine access.
    
    4. Copy the authorization code from the browser and paste it in the terminal.
    
    
    For more details, visit: https://developers.google.com/earth-engine/guides/python_install
    


Manual authentication failed: Not signed up for Earth Engine or project is not registered. Visit https://developers.google.com/earth-engine/guides/access
Service account initialization failed: Caller does not have required permission to use project ee-epstm2024. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=ee-epstm2024 and then retry. Propagation of the new permission may take a few minutes.


Earth Engine initialized with service account successfully!
Initialized: True
Authenticated: True
Project: projects/ee-epstm2024/assets/Reference_data_sumsel_test


# Module 1: Acquisition of Near-Cloud-Free Satellite Imagery

### Shapefile Validation Testing

In [3]:
# Import the validators
from luma_ge.input_utils import shapefile_validator, kml_validator
import geopandas as gpd

In [4]:
# #Initialize shapefile validator
# print("=" * 60)
# print("SHAPEFILE VALIDATION TEST")
# print("=" * 60)
# #initilizae the validator
# validator_shp = shapefile_validator(verbose=True)
# #Path to test shapefile. can be change accordingly
# shapefile_path = '../data/area_of_interest.shp'
# #for demo, used geopandas to load the shapefile
# test_gdf = gpd.read_file(shapefile_path)
# print(f"\nLoaded shapefile with {len(test_gdf)} features")
# print(f"Geometry types: {test_gdf.geometry.geom_type.unique()}")

# #Validate and fix geometry
# print("\n--- Starting Validation ---")
# validated_gdf = validator_shp.validate_and_fix_geometry(test_gdf, geometry="mixed")

# if validated_gdf is not None:
#     print(f"\n✓ Validation successful!")
#     print(f"Features after validation: {len(validated_gdf)}")
#     print(f"All geometries valid: {validated_gdf.geometry.is_valid.all()}")
# else:
#     print(f"\n✗ Validation failed!")

In [5]:
# Initialize KML validator
# print("\n" + "=" * 60)
# print("KML/KMZ VALIDATION TEST")
# print("=" * 60)
# #initilizate the class
# validator_kml = kml_validator(verbose=True)
# kml_path = '../data/AOI_Kab_PagarAlam.kmz'  #Can be change accordingly
# #load the data
# try:
#     #perform the validation
#     validated_kml_gdf = validator_kml.load_and_validate(kml_path, geometry="mixed")
    
#     if validated_kml_gdf is not None:
#         print(f"\n✓ KML validation successful!")
#         print(f"Features after validation: {len(validated_kml_gdf)}")
#         print(f"CRS: {validated_kml_gdf.crs}")
#         print(f"All geometries valid: {validated_kml_gdf.geometry.is_valid.all()}")
#     else:
#         print(f"\n✗ KML validation failed!")
        
# except FileNotFoundError:
#     print(f"⚠ KML file not found at {kml_path}")
#     print("To test KML validation, provide a valid KML file path")
# except Exception as e:
#     print(f"⚠ Error testing KML: {e}")

### System Response 1.1: Area of Interest Definition

In [6]:
import geemap
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image
from luma_ge.helpers import get_aoi_from_gaul

In [7]:
#Set the country and province for the AOI using GAUL admin boundaries
#aoi = get_aoi_from_gaul(country="Indonesia", province="Sumatera Selatan")
#Alternatively, used geemap_shp_to_ee to directly used shapefile in your local machine

In [8]:
#alternatively, you can also select smaller AOI using regency data below
import pandas as pd
indo_regency = ee.FeatureCollection('projects/ee-agilakbar/assets/Indonesian_Regency')
#check Regency List
regency = indo_regency.aggregate_array("WADMKK").getInfo()
#print(pd.DataFrame(regency, columns=["WADMKK"]))
#province = indo_regency.aggregate_array("WADMPR").getInfo()
#Example for Pagar Alam
regency_name = "Kota Pagar Alam"
#Filter the FeatureCollection, used it for AOI
aoi = indo_regency.filter(ee.Filter.eq("WADMKK", regency_name)).geometry()

### System Response 1.2: Search and Filter Imagery
The EPISTEM source code supports Landsat mission data, ranging from Landsat 1 to Landsat 9. For Landsat 1 - 3, the avaliable data is corrected radiance reflectance. The Landsat 5-9 used here is collection 2 surface reflectance (SR) analysis ready data.

The retrival logic used here is as follow:
1. Retrive multispectral bands (band 1 - 7) from landsat collection 2 SR data (if avaliable)
2. Retrive thermal band from landsat collection 2 TOA data 
3. Create temporal composite for each data 
4. Stacked the final two data into a earth engine image (ee.image)

In [9]:
#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2017-01-01'
end = '2017-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
#Add the data to the map
Map = geemap.Map()
Map.addLayer(mosaic_landsat, l8_sr_visparam, 'L8 SR Mosaic')
Map.addLayer(median_landsat, l8_sr_visparam, 'L8 SR Median')
Map.addLayer(landsat_data, l8_sr_visparam, 'L8 SR Image Collection')
# set center of the map in the area of interest
Map.centerObject(aoi, 7)

2026-03-24 09:47:54,138 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-03-24 09:47:54,140 - final_Image - INFO - final_Image creation initialized.
2026-03-24 09:47:54,141 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-03-24 09:47:54,143 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-03-24 09:47:54,144 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-03-24 09:47:54,145 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-03-24 09:47:54,147 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-03-24 09:47:54,149 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-03-24 09:47:54,935 - final_Image - INFO - Creating quality mosaic from 5 images using NDVI as quality metric
2026-03-24 09:47:54,939 - final_Image - INFO - Quality mosaic created covering AOI with best available pixels
2026

In [10]:
#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()
#visualize the thermal bands and multispectral bands
Map.addLayer(median_thermal, thermal_vis, "Thermal Bands")
Map

2026-03-24 09:48:07,358 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-03-24 09:48:07,360 - Reflectance_Data - INFO - Starting thermal data fetch for Landsat 8 Top-of-atmosphere reflectance
2026-03-24 09:48:07,361 - Reflectance_Data - INFO - Date range: 2017-01-01 to 2017-12-31
2026-03-24 09:48:07,363 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-03-24 09:48:07,364 - Reflectance_Data - INFO - Fast mode enabled - detailed statistics will not be computed
2026-03-24 09:48:07,369 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for detailed info)
2026-03-24 09:48:07,787 - final_Image - INFO - Creating Median composite from 5 images
2026-03-24 09:48:07,790 - final_Image - INFO - Composite clipped to AOI
2026-03-24 09:48:09,537 - final_Image - INFO - Composite created from 2017-05-05 to 2017-08-09


Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

### Image retrival report (optional)

In [11]:
# #intialize the statistic class
# stats = Reflectance_Stats()
# #get the retrival report and automatically print them
# retrival_report = stats.get_collection_statistics(landsat_data, print_report=True)

### System Response 1.3: Imagery Download

In [12]:
# export_task = ee.batch.Export.image.toDrive(
#     image=stacked_landsat,
#     description='Landsat_Median_composite_2017_Sumsel',
#     folder='Earth Engine',
#     fileNamePrefix='Landsat_Median_composite_2017_Sumsel',
#     scale=30,
#     region=aoi,  # or aoi.geometry()
#     maxPixels=1e13
# )
# export_task.start()
# import time

# while export_task.active():
#     print('Exporting... (status: {})'.format(export_task.status()['state']))
#     time.sleep(10)

# print('Export complete (status: {})'.format(export_task.status()['state']))

# Module 2:  Land-cover classification Scheme
Three approach are provided to handle classification scheme:
1. Upload a csv file 
2. Manual input the classification scheme
3. Use default classification scheme (RESTORE+ project)

### Import the module

In [13]:
from luma_ge.classification_scheme import LULC_Scheme_Manager
#Initialize the LULC Scheme Manager
manager = LULC_Scheme_Manager()
print("Land Cover Classification Scheme Manager initialized!")
print(f"Current class count: {manager.get_class_count()}")
#Temporary function to display the classiifcation scheme in notebook
#Display current classification scheme
def display_classification_scheme(manager):
    """Display the current classification scheme in a readable format"""
    if not manager.has_classes():
        print("No classes defined yet.")
        return
    
    print("\n=== Current Classification Scheme ===")
    df = manager.get_dataframe()
    print(df.to_string(index=False))
    
    return df

# Display the scheme
df = display_classification_scheme(manager)

Land Cover Classification Scheme Manager initialized!
Current class count: 0
No classes defined yet.


### System Response 2.1a: Upload Classification Scheme

In [14]:
# import pandas as pd
# #Reset manager for CSV upload example
# manager = LULC_Scheme_Manager()
# #path to csv 
# csv_path = "../data/Example_Classification_scheme.csv"

# print("=== CSV Upload Process ===")

# # Load the CSV
# df = pd.read_csv(csv_path, sep=None, engine="python")
# print("Loaded CSV:")
# print(df)

# # Auto-detect columns
# id_col, name_col, color_col = manager.auto_detect_csv_columns(df)
# print(f"\nAuto-detected columns:")
# print(f"ID column: {id_col}")
# print(f"Name column: {name_col}")
# print(f"Color column: {color_col}")

In [15]:
# # Process CSV upload
# success, message = manager.process_csv_upload(df, id_col, name_col, color_col)
# if success:
#     print(f"✅ {message}")
    
#     # Finalize the upload
#     success, message = manager.finalize_csv_upload()
#     if success:
#         print(f"✅ {message}")
#     else:
#         print(f"❌ {message}")
# else:
#     print(f"❌ {message}")

# # Display the loaded scheme
# display_classification_scheme(manager)

### System Response 2.1b: Manual Scheme Definition

In [16]:
# #Reset manager for manual input example
# manager = LULC_Scheme_Manager()
# #Manually add the class
# print("=== Manual Class Addition ===")

# #Example of class to add
# classes_to_add = [
#     (1, "Hutan Lahan Kering", "#0E6D0E"),
#     (2, "Pertanian Lahan Kering", "#E8F800"),
#     (3, "Permukiman", "#F81D00"),
#     (4, "Badan Air", "#1512F3"),
#     (5, "Pertanian Lahan Basah", "#")
# ]

# for class_id, class_name, color_code in classes_to_add:
#     success, message = manager.add_class(class_id, class_name, color_code)
#     if success:
#         print(f"✅ {message}")
#     else:
#         print(f"❌ {message}")

# print(f"\nTotal classes: {manager.get_class_count()}")

In [17]:
# Example: Edit an existing class
print("=== Editing a Class ===")

# Edit the first class (index 0)
class_to_edit = manager.edit_class(0)
if class_to_edit:
    print(f"Editing class: {class_to_edit}")
    
    # Update the class with new information
    success, message = manager.add_class(1, "HUtan Lahan Rendah", "#004D00")
    if success:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")

# Display updated scheme
display_classification_scheme(manager)

=== Editing a Class ===
No classes defined yet.


### System Response 2.1c: Template Classification Scheme

In [18]:
# Reset manager for default scheme example
manager = LULC_Scheme_Manager()

print("=== Available Default Schemes ===")
default_schemes = manager.get_default_schemes()

for scheme_name, classes in default_schemes.items():
    print(f"\n{scheme_name}: {len(classes)} classes")
    for class_data in classes:
        print(f"  - ID {class_data['ID']}: {class_data['Class Name']} ({class_data['Color Code']})")

=== Available Default Schemes ===

RESTORE+ Project: 17 classes
  - ID 1: Undisturbed dry-land forest (#006400)
  - ID 2: Logged-over dry-land forest (#228B22)
  - ID 3: Undisturbed mangrove (#4169E1)
  - ID 4: Logged-over mangrove (#87CEEB)
  - ID 5: Undisturbed swamp forest (#2E8B57)
  - ID 6: Logged-over swamp forest (#8FBC8F)
  - ID 7: Agroforestry (#9ACD32)
  - ID 8: Plantation forest (#32CD32)
  - ID 9: Rubber monoculture (#8B4513)
  - ID 10: Oil palm monoculture (#FF8C00)
  - ID 11: Other monoculture (#DAA520)
  - ID 12: Grass/savanna (#ADFF2F)
  - ID 13: Shrub (#90EE90)
  - ID 14: Cropland (#FFFF00)
  - ID 15: Settlement (#FF0000)
  - ID 16: Cleared land (#D2B48C)
  - ID 17: Waterbody (#0000FF)


In [19]:
# Load the RESTORE+ default scheme
scheme_name = "RESTORE+ Project"
success, message = manager.load_default_scheme(scheme_name)

if success:
    print(f"✅ {message}")
else:
    print(f"❌ {message}")

# Display the loaded scheme
display_classification_scheme(manager)

✅ Loaded RESTORE+ Project with 17 classes

=== Current Classification Scheme ===
 ID            Land Cover Class Color Palette
  1 Undisturbed dry-land forest       #006400
  2 Logged-over dry-land forest       #228B22
  3        Undisturbed mangrove       #4169E1
  4        Logged-over mangrove       #87CEEB
  5    Undisturbed swamp forest       #2E8B57
  6    Logged-over swamp forest       #8FBC8F
  7                Agroforestry       #9ACD32
  8           Plantation forest       #32CD32
  9          Rubber monoculture       #8B4513
 10        Oil palm monoculture       #FF8C00
 11           Other monoculture       #DAA520
 12               Grass/savanna       #ADFF2F
 13                       Shrub       #90EE90
 14                    Cropland       #FFFF00
 15                  Settlement       #FF0000
 16                Cleared land       #D2B48C
 17                   Waterbody       #0000FF


,ID,Land Cover Class,Color Palette
0,1,Undisturbed dry-land forest,#006400
1,2,Logged-over dry-land forest,#228B22
2,3,Undisturbed mangrove,#4169E1
3,4,Logged-over mangrove,#87CEEB
4,5,Undisturbed swamp forest,#2E8B57
5,6,Logged-over swamp forest,#8FBC8F
6,7,Agroforestry,#9ACD32
7,8,Plantation forest,#32CD32
8,9,Rubber monoculture,#8B4513
9,10,Oil palm monoculture,#FF8C00


#### Select class of interest

In [20]:
manager = LULC_Scheme_Manager()
manager.load_default_scheme("RESTORE+ Project")
df = manager.get_dataframe()  # <- this provides the classification_df

selection = manager.store_classes_of_interest(
    scheme_name="RESTORE+ Project",
    classes_of_interest=[1]
)

print(selection)

{'scheme_name': 'RESTORE+ Project', 'classes_of_interest': [1]}


### System Response 2.2: Download classification scheme

In [21]:
print("=== Export Classification Scheme ===")
#Convert the selected  classification scheme manager to dataframe
classification_df = manager.get_dataframe()
print("Classification DataFrame:")
print(classification_df)
#Save the file
output_path = '../Selected_LC_Classification_Scheme.csv'
classification_df.to_csv(output_path, index=False)
print(f"\n✅ Classification scheme saved to: {output_path}")

=== Export Classification Scheme ===
Classification DataFrame:
    ID             Land Cover Class Color Palette
0    1  Undisturbed dry-land forest       #006400
1    2  Logged-over dry-land forest       #228B22
2    3         Undisturbed mangrove       #4169E1
3    4         Logged-over mangrove       #87CEEB
4    5     Undisturbed swamp forest       #2E8B57
5    6     Logged-over swamp forest       #8FBC8F
6    7                 Agroforestry       #9ACD32
7    8            Plantation forest       #32CD32
8    9           Rubber monoculture       #8B4513
9   10         Oil palm monoculture       #FF8C00
10  11            Other monoculture       #DAA520
11  12                Grass/savanna       #ADFF2F
12  13                        Shrub       #90EE90
13  14                     Cropland       #FFFF00
14  15                   Settlement       #FF0000
15  16                 Cleared land       #D2B48C
16  17                    Waterbody       #0000FF

✅ Classification scheme saved to: ..

# Module 3: Generate Region Of Interest
Three methods to generate ROI are supported in EPISTEM platform:
1. **Upload Training Data** - Upload your own shapefile
2. **On-screen Sampling** - Create samples using interactive map
3. **Default Reference Data** - Use Epistem's default training data

## Library Import and Setup

## System Response 3.1 Prerequisite Check

In [22]:
print("=== Checking Prerequisites ===")
#Load from previous module
#From Module 1 - AOI data
try:
    AOI = aoi
    print("✅ AOI from Module 1 is available")
    aoi_available = True
except:
    print("❌ AOI data not available, please run Module 1 first")
    aoi_available = False

#From Module 2 - Classification scheme
try:
    
    # For demonstration, create sample classification scheme
    LULCTable = classification_df
    print("✅ Classification scheme from Module 2 is available")
    print(f"   - Number of classes: {len(LULCTable)}")
    scheme_available = True
except:
    print("❌ Classification scheme not available, please run Module 2 first")
    scheme_available = False

if aoi_available and scheme_available:
    print("\n✅ All prerequisites met! You can proceed with training data collection.")
else:
    print("\n❌ Prerequisites not met. Please complete previous modules first.")

=== Checking Prerequisites ===
✅ AOI from Module 1 is available
✅ Classification scheme from Module 2 is available
   - Number of classes: 17

✅ All prerequisites met! You can proceed with training data collection.


In [23]:
# Modul 3a 
# Import modules and functions
import ee
import pandas as pd
from luma_ge.sample_data import SyncTrainData, SplitTrainData

## System Response 3.2 ROI Upload and content Verification

In [24]:
# # ----- Data Input -----
# # 1. Decision to upload data
# UploadTrainData = True # set as 'true' to upload your own training data shapefile
# # set as 'false' to either add train data by sampling on screen or use default training data

# # 2. Training data file path (if UploadTrainData is true)
# TrainVectPath  = '../data/Training_Sumsel_Data.shp'
# TrainField = 'ID' 
#         # Load and process training data
# TrainDataDict = SyncTrainData.LoadTrainData(
#             landcover_df=LULCTable,
#             aoi_geometry=AOI,
#             training_shp_path=TrainVectPath
#         )

In [25]:
# # ----- System response 3.2.a -----
# # Set class field
# TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)

# # Validate classes
# TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, 1)

#     # Check sample sufficiency
# TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)

#     # Filter by AOI
# TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

#     # Create training data table
# table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
#     training_data=TrainDataDict.get('training_data'),
#     landcover_df=TrainDataDict.get('landcover_df'),
#     class_field=TrainDataDict.get('class_field'))

# #Summary result
# vr = TrainDataDict.get('validation_results', {})

# print("=" * 70)
# print("TRAINING DATA SUMMARY")
# print("=" * 70)
# print(f"Total training points loaded     : {vr.get('total_points', 'N/A')}")
# print(f"Points after class filtering     : {vr.get('points_after_class_filter', 'N/A')}")
# print(f"Valid points (inside AOI)        : {vr.get('valid_points', 'N/A')}")
# print(f"Invalid classes found            : {len(vr.get('invalid_classes', []))}")
# print(f"Points outside AOI               : {len(vr.get('outside_aoi', []))}")
# print("=" * 70)

#     # --- Display the main table ---
# if table_df is not None and not table_df.empty:
#         display_df = table_df.copy()
#         if 'Percentage' in display_df.columns:
#             display_df['Percentage'] = display_df['Percentage'].apply(
#                 lambda x: f"{x:.2f}%" if isinstance(x, (int, float)) else x
#             )
#         display(display_df)
# else:
#         print("No valid training data available to display.")

## System Response 3.2 Default ROI

In [26]:
print(" Loading default reference training data...")
TrainEePath = 'projects/ee-rg2icraf/assets/Indonesia_lulc_Sample'
TrainField = 'kelas'

# Stopgap solution: Rename 'Land Cover Class' column to 'LULC_Type' if it exists
if 'Land Cover Class' in LULCTable.columns:
    LULCTable = LULCTable.rename(columns={'Land Cover Class': 'LULC_Type'})
    print("Column 'Land Cover Class' renamed to 'LULC_Type'")

    
try:
    print("Loading reference training data from Earth Engine...")
        
        # Load training data
    TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=AOI,
            training_ee_path=TrainEePath
        )
        
    print("Processing and validating reference data...")
        
        # Set class field
    TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
    TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=True)
        
        # Check sufficiency
    TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
    TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)
        
        # Create summary table
    table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )
        
    print("✅ Reference training data loaded and processed successfully!")
    print(f"Total samples: {total_samples}")
        
        # Display summary table
    display(table_df)
        
        # Store final training data
    TrainDataFinal = TrainDataDict.get('training_data')
        
        # Show validation results
    vr = TrainDataDict.get('validation_results', {})
    print(f"\nValidation Results:")
    print(f"- Total points loaded: {vr.get('total_points', 'N/A')}")
    print(f"- Points after class filter: {vr.get('points_after_class_filter', 'N/A')}")
    print(f"- Valid points (within AOI): {vr.get('valid_points', 'N/A')}")
    print(f"- Invalid classes: {len(vr.get('invalid_classes', []))}")
        
except Exception as e:
        print(f"❌ Error loading reference data: {e}")
        TrainDataFinal = None

2026-03-24 09:48:15,617 - luma_ge.sample_data - INFO - Loading training data from EE asset: projects/ee-rg2icraf/assets/Indonesia_lulc_Sample


 Loading default reference training data...
Column 'Land Cover Class' renamed to 'LULC_Type'
Loading reference training data from Earth Engine...


2026-03-24 09:48:19,577 - luma_ge.sample_data - INFO - Initial feature count: 1130000
2026-03-24 09:48:19,579 - luma_ge.sample_data - INFO - Filtering by AOI bounds...
2026-03-24 09:48:19,581 - luma_ge.sample_data - INFO - AOI geometry type: <class 'ee.geometry.Geometry'>
2026-03-24 09:48:22,305 - luma_ge.sample_data - INFO - Features after AOI filter: 99
2026-03-24 09:48:22,307 - luma_ge.sample_data - INFO - Converting to GeoDataFrame...
2026-03-24 09:48:23,809 - luma_ge.sample_data - INFO - Collection size: 99
2026-03-24 09:48:23,810 - luma_ge.sample_data - INFO - Collection size (99) is within normal limits, loading directly
2026-03-24 09:48:25,609 - luma_ge.sample_data - INFO - Features to convert: 99
2026-03-24 09:48:25,612 - luma_ge.sample_data - INFO - Successfully processed 99 features
2026-03-24 09:48:25,619 - luma_ge.sample_data - INFO - Unique classes in training data: [ 1  2  7  9 13 14 15]
2026-03-24 09:48:25,622 - luma_ge.sample_data - INFO - Class counts: {7: 42, 1: 26, 

Processing and validating reference data...
✅ Reference training data loaded and processed successfully!
Total samples: 99


,ID,LULC_class,Sample_Count,Percentage
0,7,Agroforestry,42,42.424242
1,1,Undisturbed dry-land forest,26,26.262626
2,14,Cropland,11,11.111111
3,2,Logged-over dry-land forest,10,10.101010
4,13,Shrub,4,4.040404
5,9,Rubber monoculture,3,3.030303
6,15,Settlement,3,3.030303



Validation Results:
- Total points loaded: 99
- Points after class filter: 99
- Valid points (within AOI): 99
- Invalid classes: 0


# Module 4: Region of Interest Separability Analysis

## Library Import and Setup

In [27]:
#Import the sample quality functions
from luma_ge.sample_data_quality import sample_quality, spectral_plotter
#if there's error in the import, uncomment the below line to install the PyCRS package
#!pip install PyCRS

## System Response 4.1 Computing Separability Analysis

In [28]:
# roi_path = '../data/Training_Sumsel_Data.shp'
# labeled_roi = geemap.shp_to_ee('../data/Training_Sumsel_Data.shp')
# # labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

# #Conduct the analysis
# analyzer = sample_quality(training_data=labeled_roi, 
#     image= stacked_landsat, 
#     class_property='ID',           # Column with numeric IDs (1, 2, 3, etc.)
#     region= aoi,
#     class_name_property='LC_Name'          # Column with names ('Forest', 'Urban', 'Water', etc.)
# )
# # Extract spectral values
# pixel_extract = analyzer.extract_spectral_values(scale=100, max_pixels_per_class=5000)
# samples_statistic = analyzer.sample_stats()
# sample_df = analyzer.get_sample_stats_df()
# display(sample_df)
# #Sample statistic

In [29]:
# #Sample statistic (pixel value extracted from the imagery)
# pixel_stats = analyzer.sample_pixel_stats(pixel_extract)
# pixel_stats_df = analyzer.get_sample_pixel_stats_df(pixel_extract)
# display(pixel_stats_df)

In [30]:
# #Perform separability Analysis (iether using Transformed Divergence or Jeffries Matutista )
# separability_analysis = analyzer.get_separability_df(pixel_extract, method='TD')
# display(separability_analysis)

In [31]:
# #Get the lowest separability
# lowest_sep = analyzer.lowest_separability(pixel_extract)
# display(lowest_sep)

In [32]:
# # Overall separability summary
# sep_summary = analyzer.sum_separability(pixel_extract)
# print("Overall Separability Statistics:")
# display(sep_summary)

## System Response 4.2 Sample Visualization

In [33]:
# #Box plot to detect outlier
# ploter = spectral_plotter(analyzer)
# box = ploter.plot_boxplot(pixel_extract)
# for fig in box:
#     fig.show()

In [34]:
# #static scatter plot
# stat_plot = ploter.static_scatter_plot(pixel_extract, x_band='NIR', y_band='RED', add_ellipse=True)

In [35]:
# #3D scatter plot
# multi_d_scater = ploter.scatter_plot_3d(pixel_extract)
# multi_d_scater

# Module 6: Land Cover Classification

## Library Import

In [36]:
from luma_ge.classification import FeatureExtraction, Generate_LULC
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## System Response 6.2 Classification

In [37]:
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, stacked_landsat, 
                            class_prop='kelas', train_ratio=0.5)


2026-03-24 09:48:35,846 - pyogrio._io - INFO - Created 99 records


Stratified Random Split Training Pixel Size: 51
Stratified Random Split Testing Pixel Size: 48


In [38]:
classifier = Generate_LULC()
print("Performing Classification...")
#Multiclass hard classification
classification_map, trained_model = classifier.hard_classification(strafied_train, class_property='kelas', image=stacked_landsat,
                                                          ntrees=300, min_leaf=2, return_model=True)
# Evaluate model performance
print("Evaluating model performance...")

try:
    accuracy_metrics = classifier.evaluate_model(
        trained_model=trained_model,
        test_data=stratified_test,
        class_property='kelas'
    )
    
    print("✓ Model evaluation completed")
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    


Performing Classification...
Evaluating model performance...
✓ Model evaluation completed


In [39]:
orig_hist = classification_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13
).getInfo()

print(orig_hist)

{'classification': {'1': 85827.52941176463, '2': 73377.44313725489, '7': 490033.92549019546}}


## System Response 6.3 Model Evaluation

In [40]:
# # Display accuracy results
# print("=== Model Performance Summary ===")
# print(f"Overall Accuracy: {accuracy_metrics['overall_accuracy']:.4f} ({accuracy_metrics['overall_accuracy']*100:.2f}%)")
# print(f"Kappa Coefficient: {accuracy_metrics['kappa']:.4f}")
# print(f"Overall G-Mean: {accuracy_metrics['overall_gmean']:.4f}")

# print("\n=== Per-Class Metrics ===")
# #Class Dataframe
# metrics_df = pd.DataFrame({
#     'Precision': accuracy_metrics['precision'],
#     'Recall': accuracy_metrics['recall'],
#     'F1-Score': accuracy_metrics['f1_scores'],
#     'G-Mean': accuracy_metrics['gmean_per_class']
# })

# # Round to 4 decimal places
# metrics_df = metrics_df.round(4)

# display(metrics_df)

In [41]:
# # Visualize confusion matrix
# confusion_matrix = np.array(accuracy_metrics['confusion_matrix'])
# plt.figure(figsize=(8, 6))
# sns.heatmap(confusion_matrix, 
#             annot=True, 
#             fmt='d', 
#             cmap='Blues')
# plt.title('Confusion Matrix')
# plt.xlabel('Predicted Class')
# plt.ylabel('Actual Class')
# plt.tight_layout()
# plt.show()

In [42]:
# # Get feature importance
# print("Analyzing feature importance...")

# try:
#     importance_df = classifier.get_feature_importance(trained_model)
#     print("✓ Feature importance analysis completed")
    
#     display(importance_df)
    
# except Exception as e:
#     print(f"❌ Error in feature importance analysis: {e}")
# # Visualize feature importance
# plt.figure(figsize=(10, 6))

# # Create bar plot
# bars = plt.bar(importance_df['Band'], importance_df['Importance'])

# # Add value labels on bars
# for bar, value in zip(bars, importance_df['Importance']):
#     plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
#              f'{value:.1f}%', ha='center', va='bottom')

# plt.title('Feature Importance by Spectral Band')
# plt.xlabel('Landsat 8 Bands')
# plt.ylabel('Importance')
# plt.xticks(rotation=45)
# plt.grid(axis='y', alpha=0.3)
# plt.tight_layout()
# plt.show()

## Classification Map Visualization

In [43]:
# === Load Classification Scheme ===
scheme = pd.read_csv("../Selected_LC_Classification_Scheme.csv", sep=None, engine="python")
classes = [str(x).strip() for x in scheme["Land Cover Class"].tolist()]
palette = [str(x).strip() for x in scheme["Color Palette"].tolist()]
ids = scheme["ID"].tolist()
legend_dict = dict(zip(classes, palette))

# === Visualization Parameters ===
vis_params = {
    "min": min(ids),
    "max": max(ids),
    "palette": palette
}

# === Create geemap Map ===
Map = geemap.Map() 
Map.centerObject(aoi, 7)
Map.addLayer(classification_map, vis_params, "LULC Classification")

# # === Add Legend ===
Map.add_legend(
    title="Land Cover Classification", 
    legend_dict=legend_dict
    )

# Display
Map


Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

### Map reclassification of default classification scheme

In [44]:
# Reclassify the map

final_map, info = classifier.reclassify_map_by_classes(
    classification_map=classification_map,
    classification_df=df,
    selected_classes=selection
)

info

{'histogram': {'1': 85827.52941176463, '999': 563411.3686274507},
 'existing_classes': [1],
 'missing_classes': [],
 'all_classes_present': True}

In [45]:
# Prepare Reclassified Visualization


classes_of_interest = selection["classes_of_interest"]
other_class_id = 999

scheme_filtered = scheme[scheme["ID"].isin(classes_of_interest)]

reclass_names = scheme_filtered["Land Cover Class"].tolist()
reclass_colors = scheme_filtered["Color Palette"].tolist()
reclass_ids = scheme_filtered["ID"].tolist()

# Add "Other"
reclass_names.append("Other")
reclass_colors.append("#BDBDBD")
reclass_ids.append(other_class_id)

# Sequential IDs for visualization
vis_ids = list(range(1, len(reclass_ids) + 1))

# Remap for visualization
vis_map = final_map.remap(reclass_ids, vis_ids)

reclass_vis_params = {
    "min": 1,
    "max": len(vis_ids),
    "palette": reclass_colors
}


# Add Reclassified Layer

Map.addLayer(
    vis_map,
    reclass_vis_params,
    "Reclassified LULC"
)

reclass_legend = dict(zip(reclass_names, reclass_colors))

Map.add_legend(
    title="Reclassified Land Cover",
    legend_dict=reclass_legend
)

Map

Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

## Use prebuild map and bypass classiciation workflow

In [46]:
generator = Generate_LULC()

# Check: was a default scheme chosen AND does a prebuilt map exist for it?
KNOWN_DEFAULT_SCHEMES = set(generator._prebuilt_registry.keys())
chosen_scheme = "RESTORE+ Project"   # whatever manager exposes

if chosen_scheme in KNOWN_DEFAULT_SCHEMES:
    # Modules 3–5 skipped entirely
    print(f"[Bypass] Loading prebuilt map for '{chosen_scheme}' year {2018}...") # Year can be dynamic based on user input in Module 1

    result = generator.classify_from_prebuilt(
        scheme_name    = chosen_scheme,
        aoi            = aoi,
        year           = 2018, # make this dynamic based on user input in Module 1
        scheme_classes = manager.classes,
    )

    final_map       = result["final_map"]
    present_classes = result["present_classes"]
    vis_params      = result["vis_params"]

    print(f"Loaded year    : {result['year_used']}")
    print(f"Classes in AOI : {[c['Class Name'] for c in present_classes]}") # safeguard check

else:
    # ── Normal RF path (Modules 3–6) ──────────────────────────────────────
    # Skip this block if a default scheme with a prebuilt map is selected
    raise NotImplementedError(
        "Custom scheme RF pipeline not yet wired up in this cell. "
        "Select a default scheme to use the prebuilt map bypass."
    )

Map = geemap.Map()
Map.centerObject(aoi, zoom=10)
Map.addLayer(final_map, vis_params, chosen_scheme)
Map

[Bypass] Loading prebuilt map for 'RESTORE+ Project' year 2018...
Loaded year    : 2018
Classes in AOI : ['Undisturbed dry-land forest', 'Logged-over dry-land forest', 'Agroforestry', 'Plantation forest', 'Rubber monoculture', 'Oil palm monoculture', 'Grass/savanna', 'Shrub', 'Cropland', 'Settlement', 'Cleared land']


Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

# Module 7: Thematic Accuracy Assessment

## System Response 7.3 Thematic Accuracy Assessment

In [47]:
from luma_ge.accuracy import Thematic_Accuracy_Assessment
#Initialize the accuracy assessment class
accuracy_assessor = Thematic_Accuracy_Assessment()
print("✓ Thematic Accuracy Assessment class initialized")
print(f"Supported metrics: {accuracy_assessor.supported_metrics}")

✓ Thematic Accuracy Assessment class initialized
Supported metrics: ['overall_accuracy', 'kappa', 'producer_accuracy', 'user_accuracy', 'f1_scores', 'confusion_matrix']


In [48]:
# validation_data = geemap.shp_to_ee("../data/Evaluation_Sumsel_data.shp") 

# # === 2. Create Assessment Object ===
# assessor = Thematic_Accuracy_Assessment()

# # === 3. Run Accuracy Assessment ===
# success, results = assessor.run_accuracy_assessment(
#     lcmap=classification_map,
#     validation_data=validation_data,
#     class_property='LULC_ID',   #Validation ID column
#     scale=30
# )

# # === 4. Display Results ===
# if success:
#     print("=== Thematic Accuracy Results ===")
#     summary = assessor.format_accuracy_summary(results)
#     print("Overall Accuracy :", summary['overall_accuracy'])
#     print("Kappa Coefficient:", summary['kappa'])
#     print("95% CI          :", summary['confidence_interval'])
#     print("Samples Used     :", summary['sample_size'])
# else:
#     print("Error:", results["error"])


## Validate default map

For Luma mock up version, after bypassing the classification scheme, the prebuilt map will not be validated. Therefore this part will not be used for mock up, but will be used for Luma protoype version with Epistem map

In [65]:
validation_asset = "users/hadicu06/IIASA/RESTORE/accuracy_assessment_samples/stratified_samples_targ2percSEOverallAcc_localExpertsInterpreters"
validation_fc = ee.FeatureCollection(validation_asset).filterBounds(aoi)
default_map = final_map # R+ map

# Quick sanity check — fetches only the count, not all features
n_features = validation_fc.size().getInfo()
print(f"✓ Classification image loaded")
print(f"✓ Validation collection loaded: {validation_asset}")
print(f"  → {n_features} validation features found")
 
if n_features == 0:
    raise ValueError(
        "Validation FeatureCollection is empty. "
        "Check the asset path and your EE permissions."
    )

✓ Classification image loaded
✓ Validation collection loaded: users/hadicu06/IIASA/RESTORE/accuracy_assessment_samples/stratified_samples_targ2percSEOverallAcc_localExpertsInterpreters
  → 1 validation features found


In [66]:
first_feature   = validation_fc.first().getInfo()
prop_names      = list(first_feature["properties"].keys())
sample_props    = first_feature["properties"]
sample_geom     = first_feature["geometry"]["coordinates"]   # [lon, lat]

print("\n── Properties ───────────────────────────────")
print(f"  Names  : {prop_names}")
print(f"  Sample : {sample_props}")
print(f"  Coords : lon={sample_geom[0]:.4f}, lat={sample_geom[1]:.4f}")

print(final_map.projection().getInfo())
print(validation_fc.first().geometry().projection().getInfo())
print(validation_fc.size().getInfo())


── Properties ───────────────────────────────
  Names  : ['class', 'confidence', 'consensus_type', 'consensus_type_meaning', 'eng_name', 'gee_class_id', 'location_id', 'name', 'point_id', 'sample_id']
  Sample : {'class': 11, 'confidence': None, 'consensus_type': None, 'consensus_type_meaning': None, 'eng_name': None, 'gee_class_id': None, 'location_id': None, 'name': None, 'point_id': None, 'sample_id': None}
  Coords : lon=103.1651, lat=-4.0231
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.0009678465381381702, 0, 94.74733685103617, 0, -0.0009678465381381702, 6.3016488098176255]}
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
1


In [57]:
class_property = 'class'   # e.g. "CLASS_ID"

raw_counts = (
    validation_fc
    .reduceColumns(
        reducer    = ee.Reducer.frequencyHistogram(),
        selectors  = [class_property]
    )
    .get("histogram")
    .getInfo()
)

# Sort by class ID and print as a small table
print(f"\n── Count per class ({class_property}) ────────────────")
print(f"  {'Class ID':<12}{'Count':>8}{'Share':>8}")
print(f"  {'─'*28}")
total = sum(raw_counts.values())
for cls_id, count in sorted(raw_counts.items(), key=lambda x: int(x[0])):
    share = count / total * 100
    print(f"  {cls_id:<12}{count:>8}{share:>7.1f}%")
print(f"  {'─'*28}")
print(f"  {'TOTAL':<12}{total:>8}")


── Count per class (class) ────────────────
  Class ID       Count   Share
  ────────────────────────────
  1                105   23.1%
  2                 93   20.4%
  3                  5    1.1%
  4                  5    1.1%
  5                 14    3.1%
  6                 14    3.1%
  7                 43    9.5%
  8                 10    2.2%
  9                 20    4.4%
  10                34    7.5%
  11                14    3.1%
  12                 8    1.8%
  13                29    6.4%
  14                33    7.3%
  15                17    3.7%
  16                 7    1.5%
  17                 4    0.9%
  ────────────────────────────
  TOTAL            455


In [60]:
mask_check = final_map.mask().reduceRegion(
    reducer = ee.Reducer.mean(),
    geometry = aoi,
    scale = 30,
    maxPixels = 1e9
)

print(mask_check.getInfo())

{'classification': 0.9987310701815354}


In [58]:
success, results = accuracy_assessor.run_accuracy_assessment(
    lcmap           = default_map,
    validation_data = validation_fc,
    class_property  = 'class',
    scale           = 100,
    confidence      = 0.95,
)
 
if not success:
    raise RuntimeError(f"Assessment failed: {results.get('error', 'unknown error')}")
 

2026-03-24 10:15:50,184 - luma_ge.accuracy - INFO - Starting accuracy assessment...
2026-03-24 10:15:56,438 - luma_ge.accuracy - INFO - Accuracy assessment completed successfully


In [ ]:
def write_report(results: dict) -> None: # alternative for format_accuracy_summary
    """
    Print a simplified plain-text accuracy report.
    Derives everything from the results dict alone — no CONFIG dependency.
    """
    # ── Derive display values from results dict ───────────────────────────────
    # class_ids and class_names are injected into results in Cell 5 below;
    # fall back to integer indices if they were never set.
    n_classes    = len(results["confusion_matrix"])
    class_ids    = results.get("class_ids",   list(range(n_classes)))
    class_names  = results.get("class_names", [str(i) for i in class_ids])

    cm           = results["confusion_matrix"]
    oa           = results["overall_accuracy"]
    ci_lo, ci_hi = results["overall_accuracy_ci"]
    conf_pct     = int(results["confidence_level"] * 100)
    row_sums     = [sum(cm[i]) for i in range(n_classes)]

    lines = []
    a = lines.append

    a("THEMATIC ACCURACY ASSESSMENT — SUMMARY")
    a(f"Validation asset: {validation_asset}")
    a(f"Class property: {class_property}")
    a(f"Scale: {results['scale']} m")
    a(f"Confidence level: {conf_pct}%")
    a(f"Total samples: {results['n_total']}")
    a("")

    a("OVERALL METRICS")
    a(f"Overall Accuracy (OA): {oa * 100:6.2f}%")
    a(f"{conf_pct}% Confidence Interval: {ci_lo * 100:.2f}% – {ci_hi * 100:.2f}%")
    a(f"Kappa Coefficient: {results['kappa']:.4f}")
    a("")

    a("PER-CLASS METRICS")
    a("ID  Class Name                  Prod. Acc  User Acc  F1 Score  Samples")
    for i in range(n_classes):
        a(
            f"{class_ids[i]:<3} {class_names[i]:<25}"
            f"{results['producer_accuracy'][i] * 100:>9.2f}%  "
            f"{results['user_accuracy'][i]     * 100:>8.2f}%  "
            f"{results['f1_scores'][i]         * 100:>8.2f}%  "
            f"{row_sums[i]:>7}"
        )
    a("")

    text = "\n".join(lines)
    print(text)


# ── Attach class labels to results before writing ─────────────────────────────
# raw_counts was built in the inspection cell; keys are strings from EE
results["class_ids"]   = sorted(int(k) for k in raw_counts.keys())
results["class_names"] = [str(i) for i in results["class_ids"]]  # replace with names if you have them

# ── Print the simplified report ───────────────────────────────────────────────
write_report(results)


THEMATIC ACCURACY ASSESSMENT — SUMMARY
Validation asset: users/hadicu06/IIASA/RESTORE/accuracy_assessment_samples/stratified_samples_targ2percSEOverallAcc_localExpertsInterpreters
Class property: class
Scale: 100 m
Confidence level: 95%
Total samples: 1

OVERALL METRICS
Overall Accuracy (OA):   0.00%
95% Confidence Interval: 0.00% – 0.00%
Kappa Coefficient: 0.0000

PER-CLASS METRICS
ID  Class Name                  Prod. Acc  User Acc  F1 Score  Samples
1   1                             0.00%      0.00%      0.00%        0
2   2                             0.00%      0.00%      0.00%        0
3   3                             0.00%      0.00%      0.00%        0
4   4                             0.00%      0.00%      0.00%        0
5   5                             0.00%      0.00%      0.00%        0
6   6                             0.00%      0.00%      0.00%        0
7   7                             0.00%      0.00%      0.00%        0
8   8                             0.00%      0